# 05. MCP Server と OAuth ― Amazon Quick から呼べるようにする

5本目では、04 で作った Agent を **外部（Amazon Quick）から呼べるように公開** します。
元教材では Cortex Code スキルの Step 5〜7 にあたる部分です。

| 作るもの | 役割 |
|---|---|
| MCP Server `SALES_OPS_MCP_SERVER` | Agent を MCP（標準プロトコル）のツールとして公開する窓口 |
| OAuth 連携 `SALES_OPS_QUICK_SUITE_OAUTH` | Quick のユーザーが Snowflake にログインするための仕組み |

**前提**：`04_cortex_agent.ipynb` を最後まで実行済みであること

> 🖼️ **図の差し込み位置：Quick から Snowflake への接続の流れ**  
> Quick（Chat Agent / Flow）→ OAuth ログイン → MCP Server → Cortex Agent → Cortex Analyst（Semantic View）／ Cortex Search（PDF）。返るのは結果のみ  
> （画像は `images/` に置き、`![Quick から Snowflake への接続の流れ](../images/xxx.png)` で差し込んでください）

## Step 0. セッションの準備

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

DB, SCHEMA = "CORTEX_QUICK_HOL_DB", "CORTEX_QUICK_HOL_SCHEMA"
HOL_ROLE, HOL_WH = "CORTEX_QUICK_HOL_ROLE", "CORTEX_QUICK_HOL_WH"

session.use_role("ACCOUNTADMIN")
session.use_warehouse(HOL_WH)
session.use_schema(f"{DB}.{SCHEMA}")
print("ロール:", session.get_current_role(), "/ ウェアハウス:", session.get_current_warehouse())

# Quick を使うリージョン（OAuth のリダイレクト先と完全一致させる必要があります）
QUICK_REGION = "us-west-2"
quick_redirect_uri = f"https://{QUICK_REGION}.quicksight.aws.amazon.com/sn/oauthcallback"

row = session.sql("""SELECT CURRENT_USER() AS U,
  REPLACE(CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME(), '_', '-') AS H""").collect()[0]
hol_user, account_host = row["U"], row["H"]
print("ユーザー          :", hol_user)
print("アカウントホスト  :", f"{account_host}.snowflakecomputing.com")
print("OAuth リダイレクト:", quick_redirect_uri)

> 💡 **アカウントホストについて**：MCP エンドポイントや OAuth の URL は `<組織名>-<アカウント名>.snowflakecomputing.com` で組み立てます。
> アカウント名に `_` が含まれる場合は `-` に置き換えます（URL で使えないため）。

## Step 1. MCP Server を作る

MCP（Model Context Protocol）は、AI エージェントと外部ツールをつなぐ標準プロトコルです。
Snowflake Managed MCP Server を使うと、**サーバーを自前で立てずに** Agent を外部へ公開できます。

- ツールの種類 `CORTEX_AGENT_RUN` で Agent を1つ公開します（外部には Agent だけを見せ、どのツールを使うかは Agent に任せる構成）
- Semantic View を直接公開する `CORTEX_ANALYST_MESSAGE` などの種類もあります

In [ ]:
%%sql -r create_mcp_server
CREATE OR REPLACE MCP SERVER CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_MCP_SERVER
  COMMENT = 'MCP Server exposing Sales Ops Agent for Amazon Quick'
  FROM SPECIFICATION $$
    tools:
      - title: "Sales Ops Agent"
        name: "sales_ops_agent"
        type: "CORTEX_AGENT_RUN"
        identifier: "CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_OPS_AGENT"
        description: "Ask questions about sales operations data (revenue, pipeline, reps, customers, products, marketing) and the internal sales reports (recommendations, benchmarks)."
  $$

## Step 2. 権限を付ける

MCP Server の USAGE だけでは、配下のツールは使えません。**Agent・Semantic View・Cortex Search の権限も必要**です（02〜04 で付与済み）。
ここでは MCP Server への USAGE を付け、3つがそろっているかをまとめて確認します。

> 💡 **元教材との違い**：AWS Workshop 5章 Step 6 では MCP Server への GRANT が2回書かれ、Agent への GRANT が抜けています。本ハンズオンでは 04 で Agent への GRANT を行っています。

In [ ]:
session.sql(f"GRANT USAGE ON MCP SERVER {DB}.{SCHEMA}.SALES_OPS_MCP_SERVER TO ROLE {HOL_ROLE}").collect()

for obj in ["MCP SERVER", "AGENT", "SEMANTIC VIEW", "CORTEX SEARCH SERVICE"]:
    name = {"MCP SERVER": "SALES_OPS_MCP_SERVER", "AGENT": "SALES_OPS_AGENT", "SEMANTIC VIEW": "SALES_OPS_SEMANTIC_VIEW", "CORTEX SEARCH SERVICE": "SALES_DOCS_SEARCH"}[obj]
    rows = session.sql(f"SHOW GRANTS ON {obj} {DB}.{SCHEMA}.{name}").collect()
    ok = any(r["grantee_name"] == HOL_ROLE for r in rows)
    print(f"{obj:<22} {name:<26} {HOL_ROLE} への付与: {'OK' if ok else '未付与'}")

In [ ]:
%%sql -r show_mcp_servers
SHOW MCP SERVERS IN SCHEMA CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA

## Step 3. 接続ユーザーの既定ロールを設定する

Quick から OAuth でログインしたとき、Snowflake は **そのユーザーのデフォルトロール／デフォルトウェアハウス** で処理します。

> ⚠️ **ハマりどころ③：トライアルのユーザーは既定ロールが ACCOUNTADMIN**
> - Quick からの処理は、セッションで切り替えたロールではなく **DEFAULT_ROLE** で行われます。
> - トライアルでサインアップしたユーザーの DEFAULT_ROLE は **ACCOUNTADMIN** です。ACCOUNTADMIN / SECURITYADMIN / ORGADMIN は OAuth でのログインが**ブロック**されるため、このままでは Quick から接続できません。
> - そこで、Quick でログインするユーザー（＝今このNotebookを実行しているユーザー）の DEFAULT_ROLE を `CORTEX_QUICK_HOL_ROLE` に変えます。

> 💡 変更後も、Snowsight 右上のロール切り替えや `USE ROLE ACCOUNTADMIN` で管理者として作業できます（次回ログイン時の初期ロールが変わるだけです）。
> このNotebookも各セルで `ACCOUNTADMIN` に切り替えて実行しているので影響ありません。元に戻す方法は末尾のクリーンアップを参照してください。

> 💡 **ハマりどころ④：ALLOWED_INTERFACES**
> ユーザーの `ALLOWED_INTERFACES` が `SNOWFLAKE_INTELLIGENCE` などに制限されていると、Quick の OAuth ログイン自体は成功するのに MCP 呼び出しが 401（`395300 Not authorized to access interface ALL`）になり、**Quick がログイン画面を繰り返し表示**します。トライアルの既定値（`ALL`）なら問題ありませんが、念のため下のセルで確認し、必要なら `ALL` に戻します。

In [ ]:
before = {r["property"]: r["value"] for r in session.sql(f'DESC USER "{hol_user}"').collect()
          if r["property"] in ("DEFAULT_ROLE", "DEFAULT_WAREHOUSE")}
print("変更前:", before)

APPLY_DEFAULT_ROLE = True   # 既定ロールを変えたくない場合は False（その場合 Quick からは接続できません）
if APPLY_DEFAULT_ROLE:
    session.sql(f'ALTER USER "{hol_user}" SET DEFAULT_ROLE = \'{HOL_ROLE}\'').collect()
    session.sql(f'ALTER USER "{hol_user}" SET DEFAULT_WAREHOUSE = \'{HOL_WH}\'').collect()
    print("変更後: DEFAULT_ROLE =", HOL_ROLE, "/ DEFAULT_WAREHOUSE =", HOL_WH)

# ALLOWED_INTERFACES の確認（SNOWFLAKE_INTELLIGENCE のみだと Quick が OAuth ループになる）
props = {r["property"]: r["value"] for r in session.sql(f'DESC USER "{hol_user}"').collect()}
cur = (props.get("ALLOWED_INTERFACES") or "[ALL]")
print("ALLOWED_INTERFACES:", cur)
if "ALL" not in cur.upper().replace("[", " ").replace("]", " ").replace(",", " ").split():
    session.sql(f"ALTER USER \"{hol_user}\" SET ALLOWED_INTERFACES = ('ALL')").collect()
    print("→ ('ALL') に戻しました（Quick の MCP 接続に必要）")

## Step 4. OAuth 連携を作る

Amazon Quick のユーザーは OAuth で Snowflake にログインします。ログイン後の処理は、そのユーザーのロールの権限内で行われます（**RBAC がそのまま効く**）。

| 設定 | 値 | 意味 |
|---|---|---|
| `OAUTH_CLIENT_TYPE` | `CONFIDENTIAL` | Client Secret を使う方式 |
| `OAUTH_REDIRECT_URI` | Quick のコールバック URL | ログイン後の戻り先。**Quick のリージョンと完全一致**させる |
| `OAUTH_ISSUE_REFRESH_TOKENS` | `TRUE` | 再ログインなしで使い続けられるようにする |
| `OAUTH_USE_SECONDARY_ROLES` | `NONE` | DEFAULT_ROLE の権限だけで動かす（MCP 向けの推奨設定）。ユーザーが ACCOUNTADMIN を持っていても、Quick 経由ではハンズオン用ロールの権限に限定されます |
| `BLOCKED_ROLES_LIST` | 既定（ACCOUNTADMIN 等） | 管理者ロールでの OAuth ログインを禁止 |
| `OAUTH_REFRESH_TOKEN_VALIDITY` | `86400` | リフレッシュトークンの有効期間（秒、1日） |

作成には ACCOUNTADMIN が必要です。

> ⚠️ このセルを再実行すると **Client ID / Secret が変わります**。Quick 側に登録した後で再実行した場合は、Quick のコネクタも作り直してください。

In [ ]:
session.sql(f"""
CREATE OR REPLACE SECURITY INTEGRATION SALES_OPS_QUICK_SUITE_OAUTH
  TYPE = OAUTH
  ENABLED = TRUE
  OAUTH_CLIENT = CUSTOM
  OAUTH_CLIENT_TYPE = 'CONFIDENTIAL'
  OAUTH_REDIRECT_URI = '{quick_redirect_uri}'
  OAUTH_ISSUE_REFRESH_TOKENS = TRUE
  OAUTH_USE_SECONDARY_ROLES = 'NONE'
  OAUTH_REFRESH_TOKEN_VALIDITY = 86400
""").collect()
print("OK: SALES_OPS_QUICK_SUITE_OAUTH を作成しました")

## Step 5. Amazon Quick に入力する値を表示する

次のセルで、Quick のコネクタ設定に必要な値をまとめて表示します。手順は `docs/02_quick_setup_guide_ja.md` を参照してください。

> ⚠️ **Client Secret は画面共有で見せないよう注意してください。** 既定では伏せ字で表示し、`SHOW_SECRET = True` にしたときだけ全文を表示します。
> Quick に貼り付けるときだけ `True` にして実行し、貼り付け後は `False` に戻して再実行してください。

> 💡 Quick で OAuth ログイン画面が出たら、**このトライアルアカウントのユーザー名とパスワード**でログインします（MFA を登録済みならその確認も求められます）。

In [ ]:
import json
SHOW_SECRET = False

sec = json.loads(session.sql("SELECT SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('SALES_OPS_QUICK_SUITE_OAUTH') AS S").collect()[0]["S"])
base = f"https://{account_host}.snowflakecomputing.com"
secret = sec["OAUTH_CLIENT_SECRET"]

quick_settings = {
    "MCP server endpoint": f"{base}/api/v2/databases/{DB}/schemas/{SCHEMA}/mcp-servers/SALES_OPS_MCP_SERVER",
    "Client ID":           sec["OAUTH_CLIENT_ID"],
    "Client secret":       secret if SHOW_SECRET else secret[:4] + "*" * (len(secret) - 4),
    "Token URL":           f"{base}/oauth/token-request",
    "Authorization URL":   f"{base}/oauth/authorize",
    "Redirect URL":        quick_redirect_uri,
}
for k, v in quick_settings.items():
    print(f"{k:<20}: {v}")

## Step 6. 設定をまとめて確認する（トラブル時にも使えます）

| 症状 | 原因 | 対処 |
|---|---|---|
| `CREATE AGENT` で `unrecognized field execution_environment` | spec のトップに書いている | `tool_resources.<ツール名>` の中に入れる |
| Agent 実行で `missing an execution environment` | ツールにウェアハウスが紐付いていない | 上と同じ |
| Agent を作り直したら権限が消えた | `CREATE OR REPLACE` は GRANT を引き継がない | GRANT をやり直す |
| Quick から「権限がない」 | DEFAULT_ROLE に権限がない | Step 2・3 を確認 |
| OAuth ログインで拒否される | ACCOUNTADMIN 等は既定でブロック | 既定ロールを `CORTEX_QUICK_HOL_ROLE` に |
| OAuth のリダイレクトエラー | `OAUTH_REDIRECT_URI` と Quick のリージョンが不一致 | `QUICK_REGION` を合わせて Step 4 を再実行 |
| Quick で 424 / タイムアウト | MCP の呼び出しは 60 秒で打ち切り | 質問を簡単にする、検証済みクエリを増やす |
| Quick がログイン画面を繰り返す（401） | ユーザーの `ALLOWED_INTERFACES` が制限されている | Step 3 のセルで `ALL` に戻す |
| Quick のログインで `invalid_client` | Client ID / Secret の貼り間違い、または OAuth 連携を作り直した | Step 5 の値を貼り直す |
| `Access denied for trial accounts` | トライアルの AI 機能が無効 | クレジットカードを登録（01 冒頭参照） |
| Quick で Snowflake の URL エラー | アカウント名の `_` をそのまま使っている | Step 5 で表示された URL（`-` に置換済み）を使う |

In [ ]:
print("=== OAuth 連携")
for r in session.sql("DESCRIBE SECURITY INTEGRATION SALES_OPS_QUICK_SUITE_OAUTH").collect():
    if r["property"] in ("ENABLED", "OAUTH_REDIRECT_URI", "OAUTH_CLIENT_TYPE", "OAUTH_USE_SECONDARY_ROLES", "BLOCKED_ROLES_LIST"):
        print(f"  {r['property']:<28} {r['property_value']}")

print("\n=== 接続ユーザーの既定設定")
for r in session.sql(f'DESC USER "{hol_user}"').collect():
    if r["property"] in ("DEFAULT_ROLE", "DEFAULT_WAREHOUSE"):
        print(f"  {r['property']:<28} {r['value']}")

print("\n=== Quick と同じロール（DEFAULT_ROLE）で MCP / Agent / Semantic View を使えるか")
session.use_role(HOL_ROLE)
try:
    for obj, name in [("MCP SERVER", "SALES_OPS_MCP_SERVER"), ("AGENT", "SALES_OPS_AGENT")]:
        session.sql(f"DESCRIBE {obj} {DB}.{SCHEMA}.{name}").collect()
        print(f"  {HOL_ROLE} で {obj} {name} を参照: OK")
    n = session.sql(f"SELECT COUNT(*) AS N FROM SEMANTIC_VIEW({DB}.{SCHEMA}.SALES_OPS_SEMANTIC_VIEW METRICS TRANSACTIONS.TOTAL_REVENUE)").collect()[0]["N"]
    print(f"  {HOL_ROLE} で Semantic View をクエリ: OK")
finally:
    session.use_role("ACCOUNTADMIN")

✅ Snowflake 側の準備はここまでです。続きは `docs/02_quick_setup_guide_ja.md` に沿って Amazon Quick 側を設定してください。

---

## クリーンアップ（ハンズオン終了後）

作成したオブジェクトを削除します。誤って実行しないよう、`RUN_CLEANUP = True` にしたときだけ実行されます。
DB・ウェアハウス・ロールまで消す場合は `DROP_BASE_OBJECTS = True` にしてください。
既定ロールを変更した場合は、`RESTORE_DEFAULT_ROLE` に元のロール名を入れると戻せます。

In [ ]:
RUN_CLEANUP = False
DROP_BASE_OBJECTS = False
RESTORE_DEFAULT_ROLE = None   # 例: "ACCOUNTADMIN"（トライアルの初期値）

if RUN_CLEANUP:
    session.use_role("ACCOUNTADMIN")
    stmts = [
        f"DROP MCP SERVER IF EXISTS {DB}.{SCHEMA}.SALES_OPS_MCP_SERVER",
        f"DROP AGENT IF EXISTS {DB}.{SCHEMA}.SALES_OPS_AGENT",
        f"DROP SEMANTIC VIEW IF EXISTS {DB}.{SCHEMA}.SALES_OPS_SEMANTIC_VIEW",
        f"DROP SEMANTIC VIEW IF EXISTS {DB}.{SCHEMA}.SALES_OPS_SV_BASIC",
        f"DROP CORTEX SEARCH SERVICE IF EXISTS {DB}.{SCHEMA}.SALES_DOCS_SEARCH",
        "DROP SECURITY INTEGRATION IF EXISTS SALES_OPS_QUICK_SUITE_OAUTH",
    ]
    if RESTORE_DEFAULT_ROLE:
        stmts.append(f'ALTER USER "{hol_user}" SET DEFAULT_ROLE = \'{RESTORE_DEFAULT_ROLE}\'')
    if DROP_BASE_OBJECTS:
        stmts += [f"DROP DATABASE IF EXISTS {DB}", f"DROP WAREHOUSE IF EXISTS {HOL_WH}", f"DROP ROLE IF EXISTS {HOL_ROLE}",
                  # load.sql で作った Git 連携（ワークスペースを Git から作った場合、先にワークスペースを削除してください）
                  "DROP DATABASE IF EXISTS QUICK_HOL_GIT_DB", "DROP API INTEGRATION IF EXISTS QUICK_HOL_GIT_INTEGRATION"]
    for s in stmts:
        session.sql(s).collect()
        print("OK:", s)
else:
    print("RUN_CLEANUP = False のため何もしていません")